# SMS Smishing Detection — Full System on Colab

Runs the **complete Python research stack** from GitHub:

| Layer | On Colab? |
|-------|-----------|
| Data collection & merge | Yes |
| Text preprocessing | Yes (inside ML pipeline) |
| TF-IDF + Logistic Regression training | Yes |
| ML + rule evaluation | Yes |
| ML vs rules comparison (McNemar) | Yes |
| Interactive UI (Gradio) | Yes |
| Next.js web app | No — run locally (`npm run dev`) |
| React Native mobile app | No — run with Expo locally |

**Runtime → Run all**

Repo: [nesbertharuna/SMS-SMISHING-PROJECT](https://github.com/nesbertharuna/SMS-SMISHING-PROJECT)

In [19]:
import os, shutil, subprocess, sys
from pathlib import Path

GITHUB_REPO = "https://github.com/nesbertharuna/SMS-SMISHING-PROJECT.git"
GITHUB_BRANCH = "main"
REPO_DIR_NAME = "SMS-SMISHING-PROJECT"
DATASET = "dataset1"  # "dataset1" | "uci"

CONTENT = Path("/content")
REPO = CONTENT / REPO_DIR_NAME
if REPO.exists():
    shutil.rmtree(REPO)

print("Cloning latest code from GitHub...")
subprocess.run(
    ["git", "clone", "--branch", GITHUB_BRANCH, "--depth", "1", GITHUB_REPO, str(REPO)],
    check=True,
)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "notebooks"))

from colab_helpers import verify_system_files, setup_repo
setup_repo(REPO)
missing = verify_system_files(REPO)
if missing:
    raise FileNotFoundError("Repo clone incomplete. Missing:\n" + "\n".join(missing))
print("All system files present.")
print("Working directory:", REPO)

Cloning latest code from GitHub...


CalledProcessError: Command '['git', 'clone', '--branch', 'main', '--depth', '1', 'https://github.com/nesbertharuna/SMS-SMISHING-PROJECT.git', '/content/SMS-SMISHING-PROJECT']' returned non-zero exit status 128.

In [ ]:
subprocess.run([sys.executable, str(REPO / "scripts/colab_bootstrap.py"), "--install-deps", "--with-ui"], check=True)
subprocess.run([sys.executable, str(REPO / "scripts/colab_bootstrap.py"), "--verify-env"], check=True)

## 1. Data collection (demo)

In [ ]:
subprocess.run([
    sys.executable, str(REPO / "scripts/collect_sms.py"), "import",
    str(REPO / "data/raw/collected_sms_template.csv"), "--dedupe",
], cwd=REPO, check=True)
subprocess.run([sys.executable, str(REPO / "scripts/collect_sms.py"), "stats"], cwd=REPO, check=True)

## 2. Prepare dataset

In [ ]:
if DATASET == "uci":
    subprocess.run([sys.executable, str(REPO / "scripts/colab_bootstrap.py"), "--download-uci"], check=True)
else:
    subprocess.run([sys.executable, str(REPO / "scripts/colab_bootstrap.py"), "--verify-dataset"], check=True)

## 3. Train ML model

In [ ]:
from colab_helpers import run_script
r = run_script(REPO, "scripts/train_full_pipeline.py", "--dataset", DATASET, "--eval-overwrite")
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("Training failed")

## 4. ML test evaluation

In [ ]:
import pandas as pd
from IPython.display import display
from colab_helpers import read_json_report

ml_metrics = read_json_report(REPO, "reports/metrics_ml_test.json")
meta = read_json_report(REPO, "ml-backend/artifacts/pipeline_lr_tfidf.meta.json")
display(pd.DataFrame([ml_metrics["metrics_pos1"]]))
print("Confusion matrix:", ml_metrics["confusion_matrix_named"])
print("Validation:", meta.get("val_metrics"))

## 5. Rule baseline evaluation

In [ ]:
r = run_script(REPO, "experiments/evaluate_rules.py")
print(r.stdout)
rules_metrics = read_json_report(REPO, "reports/metrics_rules_test.json")
display(pd.DataFrame([rules_metrics["metrics_pos1"]]))

## 6. ML vs rules comparison

In [ ]:
r = run_script(REPO, "experiments/compare_ml_vs_rules.py")
print(r.stdout)
compare = read_json_report(REPO, "reports/compare_ml_vs_rules.json")
display(pd.DataFrame(compare["metrics"]).T)
print("McNemar:", compare["mcnemar_exact"])

## 7. Classify SMS (ML + rules)

In [ ]:
from colab_helpers import analyze_both
demo = "Econet Alert: Your line will be disconnected. Verify at http://econet-security-update.com"
out = analyze_both(REPO, demo)
print("=== ML ===\n", out["ml"])
print("\n=== RULES ===\n", out["rules"])

## 8. Interactive UI (Gradio — web-app-style demo)

In [ ]:
import gradio as gr
from colab_helpers import analyze_both

def ui_classify(text):
    if not str(text).strip():
        return "Enter an SMS message.", "Enter an SMS message."
    result = analyze_both(REPO, text)
    return result["ml"], result["rules"]

gr.Interface(
    fn=ui_classify,
    inputs=gr.Textbox(label="SMS message", lines=4),
    outputs=[gr.Markdown(label="ML"), gr.Markdown(label="Rules")],
    title="SmishUI — Colab Demo",
).launch(share=True)

## 9. Save to Google Drive (optional)

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/smishing_colab_outputs")
    dest.mkdir(parents=True, exist_ok=True)
    for rel in ["ml-backend/artifacts/pipeline_lr_tfidf.joblib", "reports/metrics_ml_test.json", "reports/compare_ml_vs_rules.json"]:
        src = REPO / rel
        if src.exists():
            shutil.copy2(src, dest / src.name)
            print("Saved", dest / src.name)